# 문자·도메인 공동 학습 — message-context-v4-public
**복원 버전: 2026-09-15 / 일반 대화 포함 · 유사 그룹 분할 · 제거 실험 · 독립 평가**
이 제목이 보이지 않으면 이전 편집본입니다. 디스크에서 파일을 다시 열고 커널을 재시작한 뒤 처음부터 실행하세요.
기존 모델을 덮어쓰지 않고 `experiments/message-context-<시각>/candidate`에 저장합니다.
NORMAL=0 / RISK=1 이진 분류기이며 유료 LLM을 호출하지 않습니다. 원본 class 1(일반 대화), 3(배송 안내)은 정상, class 2는 위험입니다.
최종 정책은 개인정보·인증정보 또는 송금 요구를 최소 주의로 처리합니다. 공동 입력 모델의 고위험 점수와 행동 요구 문맥은 미확인 URL의 간접 유도 판정을 보완합니다. 점수만으로 안전 URL·URL 없는 문자를 위험으로 올리지 않습니다.

공개 자료 보완: train-hard 33건, valid-hard 16건, test-hard 12건은 공식 자료에 근거한 재작성 예시입니다. 실제 수신 SMS나 사람 검수 데이터로 주장하지 않습니다. 출처별 분할과 provenance는 training/evaluation_sets/public_sources.json에 기록합니다. test-temporal은 미평가입니다. 체크포인트 점수는 Macro F1에서 검증 hard 정상 오탐률을 뺀 값이며 임계값은 valid+valid-hard에서만 선택합니다.

## 1. 실행 환경
프로젝트 전체가 있는 루트에서 Python 3.12 커널을 사용합니다. 필요한 패키지는 해당 커널의 Python으로 설치하세요.
`python -m pip install torch transformers datasets accelerate scikit-learn pandas huggingface-hub nbformat`
현재 환경의 CUDA에 맞는 PyTorch가 필요합니다. 자동 패키지 제거·Colab 전용 마운트는 하지 않습니다.

In [1]:
from pathlib import Path
import sys
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "training/evaluation_protocol.py").is_file():
    raise RuntimeError("PROJECT_DIR을 이 노트북과 training 폴더가 있는 실제 프로젝트 경로로 지정하세요.")
sys.path[:0] = [str(PROJECT_DIR / "training"), str(PROJECT_DIR / "ocr-service")]
WORKFLOW_VERSION = "message-context-v4-public"
USE_CPU = False
USE_KR_CURATED = True
RUN_ABLATIONS = False  # True: 동일 초기 모델로 5개 모드 각각 1에폭 학습
EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 1e-5
MAX_FPR = 0.01
TRAIN_COLLECTION_CUTOFF = None  # temporal 평가에 필요한 실제 학습 데이터 수집 종료일
EXTERNAL_DIR = PROJECT_DIR / "training/evaluation_sets"
print("실행 버전:", WORKFLOW_VERSION)
print("프로젝트 경로:", PROJECT_DIR)
HARD_TRAIN_REPEAT = 8  # recorded train-only oversampling of a small supplemental set


실행 버전: message-context-v4-public
프로젝트 경로: C:\Users\SSAFY\Desktop\smishing-checker-main\smishing-checker-main


In [2]:
import hashlib, json, re, unicodedata
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from transformers import set_seed
SEED = 42
MODEL_ID = "monologg/koelectra-base-v3-discriminator"
DATASET_ID = "meal-bbang/Korean_message"
MAX_LENGTH = 256
ID2LABEL = {0: "NORMAL", 1: "RISK"}
LABEL_MAP = {1: 0, 2: 1, 3: 0}
RUN_ID = "message-context-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
set_seed(SEED)
environment = {name: version(name) for name in ["torch", "transformers", "huggingface-hub", "datasets", "accelerate", "scikit-learn", "pandas", "numpy"]}
print(environment)
print("라벨 매핑:", LABEL_MAP)
print("실험 이름:", RUN_ID)

{'torch': '2.11.0+cu128', 'transformers': '4.57.1', 'huggingface-hub': '0.36.2', 'datasets': '4.0.0', 'accelerate': '1.14.0', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'numpy': '2.3.5'}
라벨 매핑: {1: 0, 2: 1, 3: 0}
실험 이름: message-context-20260915T082846059182Z


In [3]:
print("CUDA available:", torch.cuda.is_available())
if not USE_CPU and not torch.cuda.is_available():
    raise RuntimeError("GPU를 사용할 수 없습니다. nvidia-smi로 확인 후 복구하세요. CPU 학습은 USE_CPU=True로 명시하세요.")
RUN_DIR = PROJECT_DIR / "experiments" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
SPLIT_DIR = RUN_DIR / "splits"
SPLIT_DIR.mkdir()
MODEL_DIR = RUN_DIR / "candidate"
print("출력 경로:", RUN_DIR)
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print("GPU : ", GPU_NAME)
print("GPU Memory :", GPU_MEMORY_GB)

CUDA available: True
출력 경로: C:\Users\SSAFY\Desktop\smishing-checker-main\smishing-checker-main\experiments\message-context-20260915T082846059182Z
GPU :  NVIDIA GeForce RTX 5060 Ti
GPU Memory : 15.92828369140625


## 2. 데이터 출처 고정
사용한 모델·데이터 revision을 기록합니다. class 3의 생성 방식과 개별 라벨 신뢰도는 별도 검토가 필요합니다. 원본 라벨을 자동으로 확정된 실제 피해 여부로 간주하지 않습니다.

In [4]:
from huggingface_hub import HfApi
hub = HfApi()
DATASET_REVISION = hub.dataset_info(DATASET_ID).sha
MODEL_REVISION = hub.model_info(MODEL_ID).sha
print("Dataset revision:", DATASET_REVISION)
print("Model revision:", MODEL_REVISION)

Dataset revision: 733664490a9a1bc985e7590f8a9bd392566e6834
Model revision: 68b30cd259f34a4b5aa8786392612ba2a2617fcc


## 3. 원본 정제와 라벨 충돌 검사
일반 대화를 제외하지 않습니다. 정확히 같은 문장의 라벨 충돌은 따로 기록하고 제외합니다. 원문과 출처 라벨을 보존합니다.

In [5]:
from datasets import load_dataset
import pandas as pd
import re, unicodedata
LABEL_MAP = {1: 0, 2: 1, 3: 0}
LABEL_NAMES = {0: "일반", 1: "스미싱 위험"}
def normalize_text(value):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", value)).strip() if isinstance(value, str) else ""
dataset = load_dataset(DATASET_ID, revision=DATASET_REVISION)
df = dataset["train"].to_pandas()[["content", "class"]].rename(columns={"content": "text", "class": "source_label"})
df["source_row"] = df.index
df["source"] = DATASET_ID
df["raw_text"] = df["text"]
clean_df = df.loc[df.source_label.isin(LABEL_MAP)].copy()
clean_df["text"] = clean_df.text.map(normalize_text)
clean_df = clean_df.loc[clean_df.text.ne("")].copy()
clean_df["label"] = clean_df.source_label.map(LABEL_MAP).astype(int)
conflict_counts = clean_df.groupby("text").label.nunique()
conflict_texts = conflict_counts[conflict_counts > 1].index
clean_df.loc[clean_df.text.isin(conflict_texts)].to_csv(RUN_DIR / "label_conflicts.csv", index=False, encoding="utf-8-sig")
clean_df = clean_df.loc[~clean_df.text.isin(conflict_texts)].drop_duplicates("text").reset_index(drop=True)
clean_df["category"] = clean_df.label.map(LABEL_NAMES)
assert set(clean_df.label) == {0, 1}
clean_df.to_csv(RUN_DIR / "clean_messages.csv", index=False, encoding="utf-8-sig")
print("원본/정제 후:", len(df), len(clean_df))
print(clean_df.groupby(["source_label", "label"]).size())

원본/정제 후: 19009 15816
source_label  label
1             0        5776
2             1        7631
3             0        2409
dtype: int64


## 4. 유사 문장 그룹을 만든 뒤 분할
URL·숫자·고객 이름을 정규화하고 문자 3-gram Jaccard 0.9 이상의 연결 성분을 그룹으로 만듭니다. 연결 성분의 양 끝은 서로 덜 유사할 수도 있습니다.
라벨은 그룹 생성에 사용하지 않습니다. 그룹을 약 80/10/10으로 분할하며 실제 행 비율은 달라질 수 있습니다.

In [6]:
from sklearn.model_selection import StratifiedGroupKFold
from evaluation_protocol import near_duplicate_groups, assert_split_independence
clean_df["group_id"] = near_duplicate_groups(clean_df.text.tolist(), threshold=0.9)
expected_labels = {0, 1}
chosen = None
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
for train_idx, rest_idx in outer.split(clean_df, clean_df.label, clean_df.group_id):
    train_part, rest = clean_df.iloc[train_idx], clean_df.iloc[rest_idx]
    if set(train_part.label) != expected_labels or set(rest.label) != expected_labels:
        continue
    if rest.groupby("label").group_id.nunique().min() < 2:
        continue
    inner = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED)
    for valid_idx, test_idx in inner.split(rest, rest.label, rest.group_id):
        parts = {"train": train_part, "valid": rest.iloc[valid_idx], "test": rest.iloc[test_idx]}
        if all(set(part.label) == expected_labels for part in parts.values()):
            chosen = parts
            break
    if chosen is not None:
        break
if chosen is None:
    raise ValueError("양쪽 클래스가 포함된 그룹 분할이 불가능합니다. 그룹 크기와 데이터 구성을 검토하세요.")
splits = {name: frame.reset_index(drop=True).copy() for name, frame in chosen.items()}
for left, right in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    assert set(splits[left].group_id).isdisjoint(splits[right].group_id)
for name, frame in splits.items():
    frame.to_csv(SPLIT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
print({name: len(frame) for name, frame in splits.items()})

{'train': 12841, 'valid': 1486, 'test': 1489}


## 5. 검토된 보조 데이터 병합
간접 유도·기관/URL 쌍은 고정 split을 유지합니다. KR-MOB는 검토된 accepted.csv만 train에 추가합니다(CC BY-NC 4.0).
학습용 정상 사례 train-hard.csv와 평가용 test-hard.csv는 별개입니다. 외부 파일의 형식은 training/evaluation_sets/README.md를 확인하세요.
금융 정상 합성 데이터는 training/financial_normal_synthetic.csv에서 한 번만 병합하며, 생성 근거와 한계는 manifest에 기록합니다.

In [7]:
from finetune_indirect import prepare, SEEDS, BRAND_SEEDS
from evaluation_protocol import load_external, assert_split_independence
from smishing_api.config import DOMAIN_SEED_PATH, OFFICIAL_DOMAINS_PATH
kr_path = PROJECT_DIR / "training/kr_mob_filtered/accepted.csv" if USE_KR_CURATED else None
merged = prepare(SPLIT_DIR, kr_path=kr_path)
splits = {name: pd.DataFrame(rows) for name, rows in merged.items()}
hard_path = EXTERNAL_DIR / "train-hard.csv"
hard_train = load_external(hard_path, "hard")
if hard_train is not None:
    splits["train"] = pd.concat([splits["train"], hard_train.assign(slice="hard_normal")], ignore_index=True)
synthetic_path = PROJECT_DIR / "training/financial_normal_synthetic.csv"
financial_synthetic = load_external(synthetic_path, "financial synthetic")
if financial_synthetic is not None:
    # Synthetic rows are included once. Only the small curated hard set is repeated later.
    splits["train"] = pd.concat([splits["train"], financial_synthetic.assign(slice="financial_synthetic")], ignore_index=True)
valid_hard = load_external(EXTERNAL_DIR / "valid-hard.csv", "hard")
assert_split_independence({**splits, **({"valid_hard": valid_hard} if valid_hard is not None else {})})
train_df, valid_df, test_df = (splits[name] for name in ["train", "valid", "test"])
for name, frame in splits.items():
    frame.to_csv(SPLIT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
split_summary = {name: {"count": len(frame), "normal": int(frame.label.eq(0).sum()), "risk": int(frame.label.eq(1).sum())} for name, frame in splits.items()}
source_files = [SEEDS, BRAND_SEEDS, DOMAIN_SEED_PATH, OFFICIAL_DOMAINS_PATH]
if kr_path is not None: source_files.append(kr_path)
if hard_train is not None: source_files.append(hard_path)
if financial_synthetic is not None:
    source_files.extend([synthetic_path, PROJECT_DIR / "training/financial_normal_synthetic_manifest.json"])
if valid_hard is not None: source_files.append(EXTERNAL_DIR / "valid-hard.csv")
source_sha256 = {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in source_files}
print(split_summary)

{'train': {'count': 13212, 'normal': 7113, 'risk': 6099}, 'valid': {'count': 1494, 'normal': 766, 'risk': 728}, 'test': {'count': 1497, 'normal': 657, 'risk': 840}}


## 6. 학습 전 편향 검사와 제거 실험
길이·배송 표현·URL만 사용하는 얕은 분류기를 train에서 학습하고 valid에서 평가합니다. 높은 점수는 문맥 이외의 쉬운 구별 방법이 있다는 경고입니다.
제거 실험은 동일 초기 모델·고정 분할로 다섯 번 학습합니다. 기본값은 미실행이며 RUN_ABLATIONS=True로 켭니다. 점수 유지 하나만으로 누수 원인을 확정하지 않습니다.

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
def surface_features(frame):
    text = frame.text.astype(str)
    return pd.DataFrame({"length": text.str.len(), "delivery": text.str.contains(r"배송|택배|배달").astype(int), "url": text.str.contains(r"https?://|www\.").astype(int)})
quality = {"counts": split_summary, "baseline": {}}
for columns in [["length"], ["length", "delivery", "url"]]:
    baseline = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=SEED)
    baseline.fit(surface_features(train_df)[columns], train_df.label)
    predicted = baseline.predict(surface_features(valid_df)[columns])
    scores = {"accuracy": accuracy_score(valid_df.label, predicted), "macro_f1": f1_score(valid_df.label, predicted, average="macro")}
    quality["baseline"]["+".join(columns)] = scores
    print(columns, scores)
    if scores["macro_f1"] >= 0.98: print("표면 특징만으로 점수가 높습니다. 배포 성능으로 해석하지 마세요.")
(RUN_DIR / "data_quality.json").write_text(json.dumps(quality, ensure_ascii=False, indent=2), encoding="utf-8")

['length'] {'accuracy': 0.8453815261044176, 'macro_f1': 0.8453149266609146}
['length', 'delivery', 'url'] {'accuracy': 0.8587684069611781, 'macro_f1': 0.8581705342371886}


497

In [9]:
if RUN_ABLATIONS:
    from run_ablations import run_ablations
    ablation_report = run_ablations(MODEL_ID, MODEL_REVISION, splits, RUN_DIR / "ablations", use_cpu=USE_CPU, seed=SEED)
    print(ablation_report["modes"])
else:
    print("Ablations not executed. RUN_ABLATIONS=True enables five fresh one-epoch runs.")

Ablations not executed. RUN_ABLATIONS=True enables five fresh one-epoch runs.


## 7. 문자·도메인 입력, 평가 지표와 학습
본문과 도메인 관계는 서빙의 encode_message와 동일하게 인코딩합니다. 총 256토큰 중 관계 정보에 최대 96토큰을 확보합니다.
최대 5에폭, 200스텝 이내 간격 평가, early stopping patience 3을 사용합니다. Macro F1로 체크포인트를 고르고 ROC-AUC/AP·오탐률·재현율을 함께 기록합니다. 재현율만으로 선택하지 않습니다.

In [10]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from smishing_api.message_context import INPUT_SCHEMA, encode_message, domain_context
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, revision=MODEL_REVISION, num_labels=2, id2label=ID2LABEL, label2id={v: k for k, v in ID2LABEL.items()})
model.config.smishing_input_schema = INPUT_SCHEMA
# Duplicate only inside the TRAIN tensor dataset; frozen unique CSVs remain unchanged.
repeat_count = globals().get("HARD_TRAIN_REPEAT", 8)
hard_rows = train_df.loc[train_df["slice"].eq("hard_normal")]
train_for_model = pd.concat([train_df] + [hard_rows] * (repeat_count - 1), ignore_index=True)
model_frames = {**splits, "train": train_for_model}
if valid_hard is not None: model_frames["valid_hard"] = valid_hard
raw_datasets = DatasetDict({name: Dataset.from_pandas(frame[["text", "label"]], preserve_index=False) for name, frame in model_frames.items()})
def tokenize(row): return encode_message(tokenizer, row["text"])
tokenized = raw_datasets.map(tokenize, remove_columns=["text"]).rename_column("label", "labels")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
truncation_rates = {}
for name, frame in splits.items():
    truncated = 0
    for text in frame.text:
        context_len = min(96, len(tokenizer(domain_context(text), add_special_tokens=False)["input_ids"]))
        body_len = len(tokenizer(text, add_special_tokens=False)["input_ids"])
        truncated += body_len > MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=True) - context_len
    truncation_rates[name] = truncated / len(frame)
for label in [0, 1]:
    position = next(i for i, value in enumerate(train_df.label) if value == label)
    print("label:", label, "decoded:", tokenizer.decode(tokenized["train"][position]["input_ids"]))
print("본문 잘림 비율:", truncation_rates)

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/13443 [00:00<?, ? examples/s]

You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Token indices sequence length is longer than the specified maximum sequence length for this model (675 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/1494 [00:00<?, ? examples/s]

Map:   0%|          | 0/1497 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

label: 0 decoded: [CLS] 언급기관 = 미확인 ; 링크관계 = 링크없음 ; 호스트 = [SEP] 밀당보다 직진이 더매력쩔음 [SEP]
label: 1 decoded: [CLS] 언급기관 = 미확인 ; 링크관계 = 링크없음 ; 호스트 = [SEP] 엄마 나 폰이 망가져서 수리맡기고 컴퓨터 문자나라로 메시지 보내고 있어 엄마 지금 바뻐? [SEP]
본문 잘림 비율: {'train': 0.010899182561307902, 'valid': 0.019410977242302542, 'test': 0.008684034736138945}


In [11]:
import math
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from evaluation_protocol import score_metrics

def compute_metrics(prediction):
    probabilities = torch.softmax(torch.as_tensor(prediction.predictions), dim=-1)[:, 1].numpy()
    values = score_metrics(prediction.label_ids, probabilities)
    if valid_hard is not None and len(prediction.label_ids) == len(valid_df) + len(valid_hard):
        hard_fpr = float((probabilities[len(valid_df):] >= .5).mean())
        values["hard_normal_false_positive_rate"] = hard_fpr
        values["checkpoint_score"] = values["macro_f1"] - hard_fpr
    else:
        values["checkpoint_score"] = values["macro_f1"]
    return {key: value for key, value in values.items() if value is not None and key != "threshold"}
EVAL_STEPS = min(200, max(1, math.ceil(len(train_df) / TRAIN_BATCH_SIZE)))
training_args = TrainingArguments(output_dir=str(RUN_DIR / "checkpoints"), learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS, weight_decay=0.01, warmup_ratio=0.1,
    eval_strategy="steps", eval_steps=EVAL_STEPS, save_strategy="steps", save_steps=EVAL_STEPS,
    load_best_model_at_end=True, metric_for_best_model="checkpoint_score", greater_is_better=True,
    save_total_limit=2, use_cpu=USE_CPU, fp16=not USE_CPU and torch.cuda.is_available(),
    logging_steps=50, report_to="none", seed=SEED, data_seed=SEED)
from datasets import concatenate_datasets
selection_dataset = concatenate_datasets([tokenized["valid"], tokenized["valid_hard"]]) if valid_hard is not None else tokenized["valid"]
trainer = Trainer(model=model, args=training_args, train_dataset=tokenized["train"], eval_dataset=selection_dataset,
    processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
manifest = {"workflow_version": "message-context-v4-public", "run_id": RUN_ID,
    "dataset_id": DATASET_ID, "dataset_revision": DATASET_REVISION,
    "model_id": str(MODEL_ID), "model_revision": MODEL_REVISION,
    "source_label_map": LABEL_MAP, "input_schema": INPUT_SCHEMA,
    "split_method": "normalized char-trigram Jaccard >=0.9 components then StratifiedGroupKFold",
    "split_summary": split_summary, "hard_train_repeat": repeat_count, "financial_synthetic_count": len(financial_synthetic) if financial_synthetic is not None else 0, "effective_train_count": len(train_for_model),
    "valid_hard_count": len(valid_hard) if valid_hard is not None else 0, "checkpoint_rule": "macro_f1 minus valid-hard FPR at 0.5", "source_sha256": source_sha256,
    "split_sha256": {name: hashlib.sha256((SPLIT_DIR / f"{name}.csv").read_bytes()).hexdigest() for name in splits},
    "max_length": MAX_LENGTH, "truncation_rates": truncation_rates, "seed": SEED, "environment": environment,
    "kr_license": "CC BY-NC 4.0; KR-MOB-SMISHING Project (2026)" if USE_KR_CURATED else None,
    "limitations": ["Source labels not individually verified", "Synthetic supplements do not represent production accuracy"]}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_DIR / "training_args.json").write_text(training_args.to_json_string(), encoding="utf-8")
trainer.train()
validation_metrics = trainer.evaluate()
print(validation_metrics)
print("선택된 체크포인트:", trainer.state.best_model_checkpoint)

Step,Training Loss,Validation Loss,Count,Tn,Fp,Fn,Tp,Accuracy,Macro F1,Risk Recall,Risk Precision,Normal False Positive Rate,Roc Auc,Average Precision,Hard Normal False Positive Rate,Checkpoint Score
200,0.353000,0.237806,1510,779,3,63,665,0.956291,0.956041,0.913462,0.995509,0.003836,0.993867,0.995466,0.125000,0.831041
400,0.047700,0.043321,1510,778,4,10,718,0.990728,0.990714,0.986264,0.994460,0.005115,0.997772,0.998378,0.125000,0.865714
600,0.025100,0.034772,1510,779,3,7,721,0.993377,0.993368,0.990385,0.995856,0.003836,0.998959,0.999087,0.125000,0.868368
800,0.013500,0.040201,1510,779,3,10,718,0.991391,0.991377,0.986264,0.995839,0.003836,0.998563,0.998873,0.125000,0.866377
1000,0.002200,0.021725,1510,779,3,5,723,0.994702,0.994695,0.993132,0.995868,0.003836,0.999326,0.999492,0.125000,0.869695
1200,0.015500,0.020509,1510,780,2,5,723,0.995364,0.995358,0.993132,0.997241,0.002558,0.999569,0.999636,0.062500,0.932858
1400,0.001600,0.026517,1510,780,2,6,722,0.994702,0.994694,0.991758,0.997238,0.002558,0.999138,0.999414,0.062500,0.932194
1600,0.000800,0.023670,1510,779,3,3,725,0.996026,0.996021,0.995879,0.995879,0.003836,0.999425,0.999546,0.125000,0.871021
1800,0.000600,0.025143,1510,779,3,4,724,0.995364,0.995358,0.994505,0.995873,0.003836,0.999453,0.999565,0.125000,0.870358


{'eval_loss': 0.02050933800637722, 'eval_count': 1510, 'eval_tn': 780, 'eval_fp': 2, 'eval_fn': 5, 'eval_tp': 723, 'eval_accuracy': 0.9953642384105961, 'eval_macro_f1': 0.9953576233139543, 'eval_risk_recall': 0.9931318681318682, 'eval_risk_precision': 0.9972413793103448, 'eval_normal_false_positive_rate': 0.0025575447570332483, 'eval_roc_auc': 0.9995687656333436, 'eval_average_precision': 0.9996362302660702, 'eval_hard_normal_false_positive_rate': 0.0625, 'eval_checkpoint_score': 0.9328576233139543, 'eval_runtime': 1.281, 'eval_samples_per_second': 1178.772, 'eval_steps_per_second': 37.471, 'epoch': 2.140309155766944}
선택된 체크포인트: C:\Users\SSAFY\Desktop\smishing-checker-main\smishing-checker-main\experiments\message-context-20260915T082846059182Z\checkpoints\checkpoint-1200


## 8. 검증 전용 임계값 선택과 독립 테스트
valid에서 정상 오탐률 1% 이하 조건의 재현율 최대 임계값을 선택해 고정합니다. 유한 검증 표본의 경험적 오탐률이지 서비스 보장이 아닙니다.
내부 test, test-hard, test-temporal은 모델·임계값 선택에 사용하지 않습니다. 실제 검토 데이터가 없으면 미평가로 기록합니다. temporal 평가에는 학습 데이터의 수집 종료일을 지정해야 합니다.

In [12]:
from evaluation_protocol import select_threshold, score_metrics, load_external, assert_split_independence
validation_prediction = trainer.predict(tokenized["valid"])
validation_probs = torch.softmax(torch.as_tensor(validation_prediction.predictions), dim=-1)[:, 1].numpy()
hard_probs = None
if valid_hard is not None:
    hard_prediction = trainer.predict(tokenized["valid_hard"])
    hard_probs = torch.softmax(torch.as_tensor(hard_prediction.predictions), dim=-1)[:, 1].numpy()
threshold_result = select_threshold(validation_prediction.label_ids, validation_probs, max_fpr=MAX_FPR,
    hard_probabilities=hard_probs, hard_categories=valid_hard.category.tolist() if valid_hard is not None else None,
    max_hard_fpr=.01, min_recall=.95)
(RUN_DIR / "threshold.json").write_text(json.dumps(threshold_result, indent=2), encoding="utf-8")
if threshold_result["threshold"] is None:
    trainer.save_model(str(MODEL_DIR))
    tokenizer.save_pretrained(str(MODEL_DIR))
    raise RuntimeError("No threshold satisfies validation recall and hard-negative FPR constraints. Candidate saved for inspection; tests are not used to tune it.")
THRESHOLD = threshold_result["threshold"]
model.config.smishing_risk_threshold = THRESHOLD
(RUN_DIR / "threshold.json").write_text(json.dumps(threshold_result, indent=2), encoding="utf-8")
prediction = trainer.predict(tokenized["test"])
risk_scores = torch.softmax(torch.as_tensor(prediction.predictions), dim=-1)[:, 1].numpy()
predicted = (risk_scores >= THRESHOLD).astype(int)
assert np.array_equal(test_df.label.to_numpy(), prediction.label_ids)
test_results = test_df.copy()
test_results["risk_score"] = risk_scores
test_results["predicted_label"] = predicted
test_results["correct"] = test_results.label.eq(predicted)
test_results.to_csv(RUN_DIR / "test_predictions.csv", index=False, encoding="utf-8-sig")
metrics = score_metrics(prediction.label_ids, risk_scores, THRESHOLD)
(RUN_DIR / "test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
slice_metrics = {}
for name in ["existing", "indirect_lure", "brand_link"]:
    frame = test_results.loc[test_results["slice"].eq(name)]
    if len(frame): slice_metrics[name] = score_metrics(frame.label, frame.risk_score, THRESHOLD)
(RUN_DIR / "slice_metrics.json").write_text(json.dumps(slice_metrics, indent=2), encoding="utf-8")
external_results, external_frames = {}, {}
for name, kind in [("test-hard", "hard"), ("test-temporal", "temporal")]:
    frame = load_external(EXTERNAL_DIR / f"{name}.csv", kind, TRAIN_COLLECTION_CUTOFF)
    if frame is None:
        external_results[name] = {"status": "not_evaluated", "reason": "No reviewed external data"}
    else: external_frames[name] = frame
if external_frames: assert_split_independence({**splits, **({"valid_hard": valid_hard} if valid_hard is not None else {}), **external_frames})
for name, frame in external_frames.items():
    external_dataset = Dataset.from_list([{"text": row.text, "labels": int(row.label)} for row in frame.itertuples()])
    encoded_external = external_dataset.map(lambda row: encode_message(tokenizer, row["text"]), remove_columns=["text"])
    external_prediction = trainer.predict(encoded_external)
    probabilities = torch.softmax(torch.as_tensor(external_prediction.predictions), dim=-1)[:, 1].numpy()
    external_results[name] = score_metrics(external_prediction.label_ids, probabilities, THRESHOLD)
    external_results[name].update(status="evaluated", origins=frame.origin.unique().tolist() if "origin" in frame else ["unspecified"], sha256=hashlib.sha256((EXTERNAL_DIR / f"{name}.csv").read_bytes()).hexdigest())
(RUN_DIR / "external_metrics.json").write_text(json.dumps(external_results, indent=2), encoding="utf-8")
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
trainer.state.save_to_json(str(RUN_DIR / "trainer_state.json"))
manifest.update(best_checkpoint=trainer.state.best_model_checkpoint, best_metric=trainer.state.best_metric,
    saved_model_dir=str(MODEL_DIR), threshold=THRESHOLD)
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_DIR / "validation_metrics.json").write_text(json.dumps(validation_metrics, indent=2), encoding="utf-8")
print("Internal test:", metrics)
print("External tests:", external_results)
print("후보 모델 저장:", MODEL_DIR)

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Internal test: {'count': 1497, 'threshold': 0.9954967498779297, 'tn': 656, 'fp': 1, 'fn': 8, 'tp': 832, 'accuracy': 0.9939879759519038, 'macro_f1': 0.9939037109400629, 'risk_recall': 0.9904761904761905, 'risk_precision': 0.9987995198079231, 'normal_false_positive_rate': 0.0015220700152207, 'roc_auc': 0.999974632166413, 'average_precision': 0.9999800274947954}
External tests: {'test-temporal': {'status': 'not_evaluated', 'reason': 'No reviewed external data'}, 'test-hard': {'count': 12, 'threshold': 0.9954967498779297, 'tn': 12, 'fp': 0, 'fn': 0, 'tp': 0, 'accuracy': 1.0, 'macro_f1': 1.0, 'risk_recall': None, 'risk_precision': None, 'normal_false_positive_rate': 0.0, 'roc_auc': None, 'average_precision': None, 'status': 'evaluated', 'origins': ['source_grounded_paraphrase'], 'sha256': '4b54fcfbb71aeb52d1f2d334a0e1013d7263207332374b00d31531d69a032e75'}}
후보 모델 저장: C:\Users\SSAFY\Desktop\smishing-checker-main\smishing-checker-main\experiments\message-context-20260915T082846059182Z\candidat

## 실행 상태와 다음 단계
이 파일의 복원 검증은 소형 임시 모델로 진행하며 전체 모델 학습 완료를 뜻하지 않습니다.
처음부터 실행하면 실제 데이터 정제·학습·평가·후보 저장이 수행됩니다. RUN_ABLATIONS=True일 때만 다섯 제거 실험이 추가 실행됩니다.
평가 후 후보를 검토하고 SMISHING_MODEL_DIR로 지정해야 서비스에 반영됩니다. test-hard/test-temporal 결과가 미평가이면 별도 실제 검토 데이터가 필요합니다.